# Notebook 1: Setup & Sample Data

## Business Context

A global industrial manufacturer receives **~15,000 enterprise support tickets per month** across multiple divisions (Industrial Adhesives, Safety & PPE, Electronics & Energy, Healthcare, Consumer). Tickets arrive via email, portal, and EDI with varying levels of urgency.

**Current state:** Manual routing by 3 FTEs with ~6-hour average time-to-route and ~18% misroute rate. Misroutes cost ~$400/ticket in rework and SLA penalties.

**Goal:** Build an AI function that automatically classifies tickets by division, priority, issue type, and routing queue — reducing time-to-route to seconds and misroute rate to <5%.

---

## What This Notebook Creates

| Object | Purpose |
|--------|--------|
| `AI_STUDIO_DEMO` database | Isolated demo environment |
| `DEMO_STATE` table | Tracks notebook completion for dependency checks |
| `SUPPORT_TICKETS` table | 15 realistic enterprise tickets across all priority levels |
| `ROUTED_TICKETS` table | Destination for classified results |

**Next notebook:** `2_function_creation.ipynb` — builds the AI classification function

In [ ]:
-- Create the demo database and set context
CREATE DATABASE IF NOT EXISTS AI_STUDIO_DEMO;
USE DATABASE AI_STUDIO_DEMO;
USE SCHEMA PUBLIC;

In [ ]:
-- Dependency tracking table (used by all notebooks to signal completion)
CREATE TABLE IF NOT EXISTS DEMO_STATE (
    NOTEBOOK_ID VARCHAR PRIMARY KEY,
    COMPLETED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

## Sample Data: Enterprise Support Tickets

We create 15 realistic tickets spanning:
- **P1-Critical** (3 tickets): Production lines down, safety incidents, thermal failures
- **P2-High** (4 tickets): MIL-SPEC compliance, healthcare quality, regulatory deadlines
- **P3-Standard** (4 tickets): TDS requests, sample requests, RMAs, application engineering
- **P4-Low** (4 tickets): Catalog requests, future project scoping, pricing disputes

In [ ]:
CREATE OR REPLACE TABLE SUPPORT_TICKETS (
    TICKET_ID VARCHAR,
    RECEIVED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    CUSTOMER_NAME VARCHAR,
    CUSTOMER_ACCOUNT_TIER VARCHAR,
    TICKET_TEXT VARCHAR
);

In [ ]:
-- P1-Critical tickets
INSERT INTO SUPPORT_TICKETS (TICKET_ID, CUSTOMER_NAME, CUSTOMER_ACCOUNT_TIER, TICKET_TEXT) VALUES
('TKT-2026-00142', 'Apex Automotive Group', 'Enterprise Premium',
 'Subject: URGENT - XT-400 bond failure on automotive trim assembly\n\nOur Tier 1 line in Monterrey has been down since 6am. The XT-400 tape is delaminating from the PP substrate on the 2026 Ranger door trim after 48hrs in the humidity chamber. We ran 3 lots (batches 2026-03-441 through 443) and all failed. This is blocking 2,200 units/day. We need your application engineering team on-site or on a call within 4 hours. Our plant manager is looping in the OEM''s VP of purchasing if we don''t have a path forward today.'),
('TKT-2026-00143', 'Coastal Safety Supply', 'Standard',
 'Subject: CRITICAL - Respirator filter failure during chemical spill response\n\nDuring a chlorine gas response at the industrial port facility yesterday, two of our workers reported breakthrough odor through 6003 OV/AG cartridges that were less than 2 hours into use. We pulled them from service immediately. These were from lot 2026-02-887. Worker medical evaluations are pending. We need immediate confirmation on whether this lot has been tested and whether a recall is warranted. OSHA has been notified. Please treat this as highest priority - lives are at stake.'),
('TKT-2026-00144', 'Meridian Electronics Corp', 'Enterprise Premium',
 'Subject: Production halt - thermal interface material failure in server rack assembly\n\nOur Austin fab has stopped production on the HPC-9000 server line. The TC-5026 thermal compound applied between the GPU dies and heatsinks is showing thermal resistance 3x spec after 72hr burn-in. We''ve scrapped 340 units today ($2.1M in WIP). Our CTO is flying in tomorrow and expects a root cause briefing. We have a $45M contract with a Tier 1 cloud provider at risk. Need application engineering and your thermal lab on a bridge call within 2 hours.');

In [ ]:
-- P2-High tickets
INSERT INTO SUPPORT_TICKETS (TICKET_ID, CUSTOMER_NAME, CUSTOMER_ACCOUNT_TIER, TICKET_TEXT) VALUES
('TKT-2026-00145', 'Defense Logistics International', 'Government/Defense',
 'Subject: MIL-SPEC compliance issue on reflective tape shipment\n\nThe retroreflective sheeting (Series 680CR) delivered on PO-DLI-2026-0392 does not meet the photometric requirements per MIL-PRF-680. Our QC lab measured 180 cd/lux/m2 vs the required 250 cd/lux/m2 minimum on the white material. We have a DoD fleet marking deadline in 3 weeks and cannot accept this shipment. Need replacement material that meets spec or a formal deviation approval from your regulatory team. Contract value $1.2M, liquidated damages clause triggers in 21 days.'),
('TKT-2026-00146', 'Summit Healthcare Network', 'Enterprise',
 'Subject: Surgical tape adhesion failure reports across 3 facilities\n\nWe are seeing a pattern of adhesion failures with SecureSkin Plus surgical tape (lot 2026-01-554) across our network. In the past week, we''ve had 12 reported incidents of tape lifting during post-operative wound care in humid environments. No patient injuries yet but our Chief Medical Officer wants this investigated immediately. We''re pulling the lot from all 47 facilities as a precaution. Need your quality team to provide a formal investigation report and lot disposition within 5 business days. FDA MedWatch report is being filed.'),
('TKT-2026-00147', 'Pacific Rim Converters Ltd', 'Distributor',
 'Subject: Quality deviation on BondPro DP420 shipment affecting our customers\n\nWe received 500 cases of DP420 epoxy adhesive (batch 2026-04-201) and our downstream customer (major aerospace OEM) is reporting cure times 40% longer than spec. They''ve rejected 3 production lots built with this material. We need an urgent quality hold investigation, COA review, and replacement product within 10 days or we lose a $3M annual account. Our sales VP is asking for escalation to your regional director.'),
('TKT-2026-00156', 'Athena Semiconductor', 'Enterprise Premium',
 'Subject: Regulatory deadline - need REACH compliance declaration for 5 materials ASAP\n\nOur EU compliance team has an audit on March 28th and we''re missing REACH SVHC declarations for the following materials we source from you: TC-5026, Novec 7100, 75 Spray Adhesive, Scotch-Weld 2216, and VHB 5952. Our compliance officer says without these declarations we risk a production stop at our Dresden fab. Can your regulatory team provide the certificates within 72 hours? This is directly tied to EU market access for $200M in annual shipments.');

In [ ]:
-- P3-Standard tickets
INSERT INTO SUPPORT_TICKETS (TICKET_ID, CUSTOMER_NAME, CUSTOMER_ACCOUNT_TIER, TICKET_TEXT) VALUES
('TKT-2026-00148', 'Greenfield Manufacturing Co', 'Standard',
 'Subject: Technical data sheet request - structural adhesives for composite bonding\n\nWe''re evaluating structural adhesives for a new carbon fiber bicycle frame assembly process. Can you send us TDS and MSDS for the following products: BondPro AF163-2, DP460NS, and 2216 B/A Gray? Also interested in any application guides for composite-to-composite bonding with surface energy below 40 mN/m. Timeline: we''re in the design phase with prototyping starting Q3.'),
('TKT-2026-00149', 'Northern Plains Electric Coop', 'End User',
 'Subject: Requesting samples of cold-shrink splice kits for 15kV underground cable\n\nWe''re planning to replace 45 direct-burial splice joints on our 15kV distribution system this summer. Currently using heat-shrink but looking at cold-shrink alternatives for faster crew installation times. Can we get 3 evaluation samples of the QS-II series for 1/0 AWG XLPE cable? Also would appreciate a field installation training session if available for our lineworkers. Project starts June 15th.'),
('TKT-2026-00150', 'Hartwell Packaging Solutions', 'Converter',
 'Subject: Standard RMA for defective transfer tape rolls\n\nWe received 20 rolls of 467MP transfer tape on PO HPK-5567 and 4 rolls have visible contamination/debris embedded in the adhesive layer. Photos attached. Requesting RMA for replacement. This is not urgent - we have sufficient inventory to continue production. Please process at standard timeline.'),
('TKT-2026-00151', 'Brightline Solar Inc', 'OEM',
 'Subject: Application engineering support for backsheet lamination adhesive\n\nWe''re developing a new solar panel design with a PVDF backsheet and need help selecting the right lamination adhesive. Requirements: UV stable 25+ years, operating temp -40C to 85C, must maintain adhesion through 1000hr damp heat testing per IEC 61215. Currently testing your 9795B tape but seeing edge delamination after 600 hours. Can your application engineering team review our lamination process parameters and suggest alternatives? Budget approved, need recommendation within 3 weeks.');

In [ ]:
-- P4-Low tickets
INSERT INTO SUPPORT_TICKETS (TICKET_ID, CUSTOMER_NAME, CUSTOMER_ACCOUNT_TIER, TICKET_TEXT) VALUES
('TKT-2026-00152', 'Valley Distributors Inc', 'Distributor',
 'Subject: 2026 product catalog and pricing update request\n\nHi team - can you send us the updated 2026 catalog and distributor price list for the Industrial Adhesives & Tapes division? We''re updating our website and internal ordering system. Also curious if there are any new products launching in Q3 that we should be aware of for our fall trade show booth. No rush on this - next 2-3 weeks is fine.'),
('TKT-2026-00153', 'TechForward Robotics', 'OEM',
 'Subject: Future project scoping - adhesive solutions for collaborative robot joints\n\nWe''re in early R&D for a next-gen cobot platform (18-month timeline to production). Looking for adhesive solutions that can bond dissimilar materials (aluminum to PEEK polymer) in articulating joints under cyclic loading (10M cycles). Is this something your advanced materials team would be interested in co-developing? We have ITAR clearance if needed. Just looking to start a conversation at this point.'),
('TKT-2026-00154', 'Consolidated Building Materials', 'End User',
 'Subject: Pricing dispute on last invoice - wrong contract price applied\n\nInvoice INV-2026-88432 shows $347/case for ProMount strips (17206-ES) but our contract CBM-2024-NATL specifies $312/case for annual volumes over 5000 cases. We''ve ordered 8,400 cases YTD. Please credit the difference ($2,940) and correct the pricing in your system going forward. Our AP team is holding payment until this is resolved.'),
('TKT-2026-00155', 'Redstone Energy Partners', 'Enterprise',
 'Subject: Warranty claim - CableGuard cable joints failing prematurely\n\nWe have 7 CableGuard 2131 resin splice joints on our 35kV transmission lines that have failed within 18 months of installation (rated life 30 years). All were installed by certified contractors following your published procedure. Failure mode appears to be moisture ingress at the cable-resin interface. Total replacement cost estimated at $89,000. Submitting warranty claim with installation records and failure analysis photos attached. Need formal response within 10 business days per warranty terms.');

In [ ]:
-- Results table for classified tickets
CREATE OR REPLACE TABLE ROUTED_TICKETS (
    TICKET_ID VARCHAR,
    TICKET_TEXT VARCHAR,
    CLASSIFICATION VARIANT,
    CLASSIFIED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

## Data Preview

Let's verify the sample data was loaded correctly.

In [ ]:
SELECT 
    TICKET_ID,
    CUSTOMER_NAME,
    CUSTOMER_ACCOUNT_TIER,
    LEFT(TICKET_TEXT, 80) AS TICKET_PREVIEW
FROM SUPPORT_TICKETS
ORDER BY TICKET_ID;

---
## Setup Complete

All tables and sample data are ready. Proceed to **Notebook 2: AI Function Creation** to build the classification function.

In [ ]:
-- Mark this notebook as completed for dependency tracking
MERGE INTO DEMO_STATE AS t
USING (SELECT 'setup' AS NOTEBOOK_ID, CURRENT_TIMESTAMP() AS COMPLETED_AT) AS s
ON t.NOTEBOOK_ID = s.NOTEBOOK_ID
WHEN MATCHED THEN UPDATE SET COMPLETED_AT = s.COMPLETED_AT
WHEN NOT MATCHED THEN INSERT (NOTEBOOK_ID, COMPLETED_AT) VALUES (s.NOTEBOOK_ID, s.COMPLETED_AT);